Material for the book [Probability and Statistics for Data Science](https://a.co/d/cAss9mO). A free preprint, videos, code, slides and solutions to exercises are available at https://www.ps4ds.net/

Monte Carlo simulations for example problem about doctors in the Australian Outback.\
Topics and relevant videos: [Election forecasting](https://youtu.be/iglSYEcWFKs?si=szm4NfGmINoiPcGA), [Bayesian parametric modeling](https://www.youtube.com/watch?v=G63ZmHg83mg), [independence, conditional independence](https://www.youtube.com/watch?v=gJNArEm5U_A), [the Monte Carlo method](https://www.youtube.com/watch?v=vIY_J85hHdw)

Author: Carlos Fernandez-Granda

In [1]:
import numpy as np
from scipy import stats

# Observed data and prior parameters
data = np.array([35, 37, 34, 23, 28, 32, 29, 31, 36, 34])
n_days = len(data)
sum_emergencies = data.sum()

alpha_prior = 1
beta_prior = 1  # Rate parameter

In [2]:
# Conjugate gamma posterior parameters
# Posterior rate beta_post = beta_prior + n_days
# Posterior shape alpha_post = alpha_prior + sum_emergencies
alpha_post = alpha_prior + sum_emergencies  # 320
beta_post = beta_prior + n_days             # 11

# ---------------------------------------------------------
# Monte Carlo simulation
# ---------------------------------------------------------
np.random.seed(2026)
num_samples = 1_000_000

# Sample lambda from gamma posterior (numpy scale = 1 / beta_post)
lambdas = np.random.gamma(shape=alpha_post, scale=1 / beta_post, size=num_samples)

# Sample daily emergencies from Poisson using sampled lambdas
simulated_emergencies = np.random.poisson(lam=lambdas)

# Determine minimum doctors needed so P(unavailability) < 0.05 => P(Y <= k) >= 0.95
simulated_doctors = int(np.percentile(simulated_emergencies, 95))

# If percentile lands precisely on exact threshold, ensure strictly <= 5% unavailability
if np.mean(simulated_emergencies > simulated_doctors) > 0.05:
    simulated_doctors += 1

sim_prob_unavailable = np.mean(simulated_emergencies > simulated_doctors)

# ---------------------------------------------------------
# Analytical Verification using scipy.stats.nbinom
# ---------------------------------------------------------
# The Gamma-Poisson mixture yields a Negative Binomial posterior predictive distribution:
# n = alpha_post, p = beta_post / (beta_post + 1)
p_param = beta_post / (beta_post + 1.0)
n_param = alpha_post

# Using Inverse CDF / Quantile Function (ppf)
analytical_doctors = int(stats.nbinom.ppf(0.95, n_param, p_param))

analytical_prob_unavailable = 1 - stats.nbinom.cdf(analytical_doctors, n_param, p_param)

# Print Results
print("--- SIMULATION RESULTS ---")
print(f"Required Doctors (Monte Carlo): {simulated_doctors}")
print(f"Probability of unavailability: {sim_prob_unavailable:.4f}")

print("\n--- ANALYTICAL RESULTS (scipy.stats.nbinom) ---")
print(f"Required Doctors (Exact): {analytical_doctors}")
print(f"Probability of unavailability: {analytical_prob_unavailable:.4f}")

--- SIMULATION RESULTS ---
Required Doctors (Monte Carlo): 39
Probability of unavailability: 0.0375

--- ANALYTICAL RESULTS (scipy.stats.nbinom) ---
Required Doctors (Exact): 39
Probability of unavailability: 0.0378
